# SPARK — SisFall Classical-ML Pipeline (Fairness/Bias & Fusemachines AIF context)

**Purpose:** a standalone 6-stage ML pipeline assignment (dataset prep → preprocessing → EDA → modeling with 2+ tuned algorithms → evaluation → predictions), built on real SisFall data already downloaded for the SPARK major project.

**Relationship to SPARK's production pipeline** (`training/train_cnn.py`, `training/data_prep/prepare_sisfall.py` in the repo):
- Stage 1 of this notebook **reuses** `prepare_sisfall.py` unmodified — it is already committed and verified end-to-end (38,426 windows / 38 of 38 subjects / 34 of 34 activity codes, see `dev_logs/SPARK_TRACKER.md` v23). This notebook does not re-implement raw-file parsing.
- Stages 4–6 here train **Random Forest + XGBoost on engineered statistical features** — a classical-ML track that is complementary to, not a duplicate of, the repo's own 1D CNN (`train_cnn.py`, which already exists and consumes raw windows directly, not engineered features).
- Rationale for the classical track: (a) it is a genuine second modeling family for this assignment's "at least 2 algorithms" requirement without redundantly re-deriving the CNN spec that's already locked in the proposal; (b) tree ensembles expose feature importances directly, which is a natural on-ramp to the SHAP explainability work already planned for SPARK's gateway (`dev_logs/SPARK_TRACKER.md`, curriculum-alignment table, Fusemachines Week 5 row); (c) it can act as a fast, interpretable sanity-check baseline before/alongside the CNN.
- This notebook's outputs (engineered feature table, tuned RF/XGB models) are additive artifacts for the assignment. They do not replace, alter, or get consumed by `train_cnn.py`.

**Binary task:** FALL (SisFall codes F01–F15) vs. NON_FALL / ADL (D01–D19) — same binary collapse `train_cnn.py` uses, for direct comparability of held-out metrics.


## 0. Setup

Run once per Colab session. Clones the SPARK repo (for `prepare_sisfall.py`) and installs the classical-ML deps this notebook needs on top of Colab's defaults.


In [ ]:
# Mount Drive — point SISFALL_ZIP_PATH below at your downloaded SisFall archive.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/Aaradhya-Dev-Tamrakar/SPARK.git /content/SPARK 2>&1 | tail -5
!pip install -q xgboost scikit-learn seaborn


In [ ]:
import sys
sys.path.insert(0, '/content/SPARK/training/data_prep')

import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid")


## 1. Dataset preparation

Reuses `prepare_sisfall.py` from the SPARK repo as-is (subprocess call, not reimplemented) — it already parses SisFall's raw per-trial `.txt` files (38 subjects, 15 fall types, 19 ADL types), downsamples 200 Hz → 100 Hz, and slices into fixed 200-sample (2 s) windows across 6 channels: `a_x, a_y, a_z, w_x, w_y, w_z`.

**Set `SISFALL_SRC` below to your extracted `SisFall_dataset/` folder** (the one containing `SA01/`, `SA02/`, ..., `SE01/`, ... subfolders) — e.g. under your mounted Drive.


In [ ]:
SISFALL_SRC = Path('/content/drive/MyDrive/SisFall_dataset')   # <-- EDIT to your actual path
OUT_DIR = Path('/content/sisfall_prepared')

assert SISFALL_SRC.is_dir(), f"SisFall source not found at {SISFALL_SRC} — edit SISFALL_SRC above."

result = subprocess.run(
    [sys.executable, '/content/SPARK/training/data_prep/prepare_sisfall.py',
     '--src', str(SISFALL_SRC), '--out', str(OUT_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("prepare_sisfall.py failed — see stderr above.")


In [ ]:
windows = np.load(OUT_DIR / 'windows.npy')       # (N, 200, 6) float32
labels_raw = np.load(OUT_DIR / 'labels.npy')      # (N,) e.g. "F03", "D11"
meta = pd.read_csv(OUT_DIR / 'meta.csv')

# Binary collapse — same convention as train_cnn.py: F* -> 1 (FALL), D* -> 0 (NON_FALL)
labels_bin = np.where(np.char.startswith(labels_raw, 'F'), 1, 0).astype(np.int32)

print(f"windows: {windows.shape}")
print(f"FALL windows:     {int(labels_bin.sum())}")
print(f"NON_FALL windows: {int((1 - labels_bin).sum())}")
print(f"subjects:         {meta['subject'].nunique()}")


## 2. Dataset preprocessing

Two things happen here, distinct from Stage 1's *parsing*:

1. **Feature engineering** — each 200×6 raw window is collapsed into a fixed-length statistical feature vector per channel (mean, std, min, max, range, RMS, signal magnitude area). Tree-based models (Stage 4) take tabular features, not raw time-series, so this step is required for them specifically — `train_cnn.py`'s CNN consumes the raw 200×6 windows directly and does not need this.
2. **Train/val/test split + scaling** — grouped by subject (no subject appears in more than one split), matching `train_cnn.py`'s own leakage-prevention approach, then `StandardScaler` fit on train only.


In [ ]:
def extract_features(window: np.ndarray) -> dict:
    """window: (200, 6) -> flat dict of per-channel statistical features."""
    channel_names = ['a_x', 'a_y', 'a_z', 'w_x', 'w_y', 'w_z']
    feats = {}
    for i, ch in enumerate(channel_names):
        col = window[:, i]
        feats[f'{ch}_mean'] = col.mean()
        feats[f'{ch}_std'] = col.std()
        feats[f'{ch}_min'] = col.min()
        feats[f'{ch}_max'] = col.max()
        feats[f'{ch}_range'] = col.max() - col.min()
        feats[f'{ch}_rms'] = np.sqrt(np.mean(col ** 2))
    # Signal Magnitude Area across the 3 accelerometer axes — standard fall-detection feature,
    # captures overall movement intensity across axes jointly rather than per-axis only.
    acc = window[:, 0:3]
    feats['sma_acc'] = np.mean(np.sum(np.abs(acc), axis=1))
    # Peak resultant acceleration — the single largest instantaneous |a| in the window,
    # directly relevant to fall detection (falls produce a sharp acceleration spike).
    resultant = np.sqrt(np.sum(acc ** 2, axis=1))
    feats['acc_resultant_peak'] = resultant.max()
    return feats

feature_rows = [extract_features(w) for w in windows]
X_features = pd.DataFrame(feature_rows)
X_features['subject'] = meta['subject'].values
X_features['label'] = labels_bin

print(f"Engineered feature table: {X_features.shape}")
X_features.head()


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

subjects = X_features['subject'].values
y = X_features['label'].values
feature_cols = [c for c in X_features.columns if c not in ('subject', 'label')]
X = X_features[feature_cols].values

# 80/10/10, grouped by subject — same no-leakage principle as train_cnn.py's subject_stratified_split.
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=RANDOM_SEED)
trainval_idx, test_idx = next(gss1.split(X, y, groups=subjects))

gss2 = GroupShuffleSplit(n_splits=1, train_size=0.9, random_state=RANDOM_SEED)  # 0.9 of the 80% -> 10% overall val
train_idx_rel, val_idx_rel = next(gss2.split(X[trainval_idx], y[trainval_idx], groups=subjects[trainval_idx]))
train_idx = trainval_idx[train_idx_rel]
val_idx = trainval_idx[val_idx_rel]

# Leakage check — assert, don't assume.
assert not (set(subjects[train_idx]) & set(subjects[test_idx]))
assert not (set(subjects[train_idx]) & set(subjects[val_idx]))
assert not (set(subjects[val_idx]) & set(subjects[test_idx]))

print(f"train: {len(train_idx)} windows, {len(set(subjects[train_idx]))} subjects")
print(f"val:   {len(val_idx)} windows, {len(set(subjects[val_idx]))} subjects")
print(f"test:  {len(test_idx)} windows, {len(set(subjects[test_idx]))} subjects")


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X[train_idx])   # fit on train only — no leakage into val/test
X_val = scaler.transform(X[val_idx])
X_test = scaler.transform(X[test_idx])

y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]


## 3. EDA (Exploratory Data Analysis)

Class balance, per-channel distributions split by class, and a correlation heatmap over the engineered features — checks the modeling stage isn't walking into a surprise (e.g. a near-duplicate feature pair, or a channel with no separating power).


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
pd.Series(y_train).map({0: 'NON_FALL', 1: 'FALL'}).value_counts().plot(kind='bar', ax=ax, color=['#4C72B0', '#C44E52'])
ax.set_title('Class balance — training set')
ax.set_ylabel('window count')
plt.tight_layout()
plt.show()

print(f"Fall ratio (train): {y_train.mean():.3f}")


In [ ]:
# Peak resultant acceleration is the single most fall-relevant engineered feature —
# a fall should show a visibly higher peak spike than ADLs.
fig, ax = plt.subplots(figsize=(7, 4))
for cls, name, color in [(0, 'NON_FALL', '#4C72B0'), (1, 'FALL', '#C44E52')]:
    mask = y_train == cls
    ax.hist(X_features.loc[train_idx, 'acc_resultant_peak'][mask], bins=40, alpha=0.6, label=name, color=color)
ax.set_xlabel('acc_resultant_peak')
ax.set_ylabel('count')
ax.set_title('Peak resultant acceleration by class')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = X_features[feature_cols].corr()
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': 0.7})
ax.set_title('Engineered feature correlation matrix')
plt.tight_layout()
plt.show()


## 4. Modeling — Random Forest & XGBoost (hyperparameter-tuned)

Two algorithms, both tuned via `GridSearchCV` on the train+val split (5-fold, grouped by subject to keep the no-leakage guarantee inside CV too, not just at the outer split).

Both are trained on the same engineered feature table from Stage 2 — directly comparable to each other, and their held-out test metrics (Stage 5) are directly comparable to `train_cnn.py`'s own reported Sensitivity/Specificity/F1/AUC-ROC targets (≥90% Sensitivity, ≥90% Specificity per the proposal).


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV

# Fold train+val together for CV-based tuning; test stays untouched until Stage 5.
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])
subjects_trainval = np.concatenate([subjects[train_idx], subjects[val_idx]])

group_kfold = GroupKFold(n_splits=5)

rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [8, 16, None],
    'min_samples_leaf': [1, 4],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1),
    rf_param_grid,
    cv=group_kfold.split(X_trainval, y_trainval, groups=subjects_trainval),
    scoring='f1',
    n_jobs=-1,
)
rf_grid.fit(X_trainval, y_trainval)

print(f"Best RF params: {rf_grid.best_params_}")
print(f"Best RF CV F1:  {rf_grid.best_score_:.4f}")
rf_model = rf_grid.best_estimator_


In [ ]:
from xgboost import XGBClassifier

# scale_pos_weight handles the FALL/ADL class imbalance for XGBoost (its analogue of RF's class_weight).
neg, pos = (y_trainval == 0).sum(), (y_trainval == 1).sum()
scale_pos_weight = neg / pos

xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.05, 0.1],
}

xgb_grid = GridSearchCV(
    XGBClassifier(
        random_state=RANDOM_SEED,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        n_jobs=-1,
    ),
    xgb_param_grid,
    cv=group_kfold.split(X_trainval, y_trainval, groups=subjects_trainval),
    scoring='f1',
    n_jobs=-1,
)
xgb_grid.fit(X_trainval, y_trainval)

print(f"Best XGB params: {xgb_grid.best_params_}")
print(f"Best XGB CV F1:  {xgb_grid.best_score_:.4f}")
xgb_model = xgb_grid.best_estimator_


## 5. Evaluation

Held-out test set (untouched by any tuning above), same metric set `train_cnn.py` reports — Sensitivity (Recall, FALL), Specificity (Recall, NON_FALL), F1, AUC-ROC — for direct comparability against the CNN and against the proposal's ≥90%/≥90% targets.


In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

def evaluate_model(model, X_test, y_test, name):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    fall_mask = y_test == 1
    nonfall_mask = y_test == 0
    sensitivity = (y_pred[fall_mask] == 1).sum() / fall_mask.sum()
    specificity = (y_pred[nonfall_mask] == 0).sum() / nonfall_mask.sum()
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    print(f"--- {name} — held-out test set ---")
    print(f"Sensitivity (Recall, FALL):     {sensitivity:.4f}  (target >= 0.90)")
    print(f"Specificity (Recall, NON_FALL): {specificity:.4f}  (target >= 0.90)")
    print(f"F1-score:                       {f1:.4f}")
    print(f"AUC-ROC:                        {auc:.4f}")

    fig, ax = plt.subplots(figsize=(4, 4))
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['NON_FALL', 'FALL']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name} — confusion matrix')
    plt.tight_layout()
    plt.show()

    return {'model': name, 'sensitivity': sensitivity, 'specificity': specificity, 'f1': f1, 'auc_roc': auc}

results = []
results.append(evaluate_model(rf_model, X_test, y_test, 'Random Forest'))
results.append(evaluate_model(xgb_model, X_test, y_test, 'XGBoost'))

results_df = pd.DataFrame(results)
results_df


In [ ]:
# Feature importance — the direct interpretability payoff of the classical track,
# and a natural precursor to the SHAP work already planned for SPARK's gateway.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model, name in [(axes[0], rf_model, 'Random Forest'), (axes[1], xgb_model, 'XGBoost')]:
    importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
    importances.plot(kind='barh', ax=ax, color='#4C72B0')
    ax.invert_yaxis()
    ax.set_title(f'{name} — top 10 feature importances')
plt.tight_layout()
plt.show()


## 6. Predictions

Runs the better of the two tuned models (by test F1) on a handful of individual held-out windows and prints a per-window verdict — the shape of the single-window inference call SPARK's gateway will eventually need, minus the actual deployment plumbing.


In [ ]:
best_row = results_df.loc[results_df['f1'].idxmax()]
best_model = rf_model if best_row['model'] == 'Random Forest' else xgb_model
print(f"Best model by test F1: {best_row['model']} (F1={best_row['f1']:.4f})")


In [ ]:
n_show = 10
sample_idx = np.random.RandomState(RANDOM_SEED).choice(len(X_test), size=n_show, replace=False)

preds = best_model.predict(X_test[sample_idx])
probs = best_model.predict_proba(X_test[sample_idx])[:, 1]
true = y_test[sample_idx]

pred_table = pd.DataFrame({
    'true_label': np.where(true == 1, 'FALL', 'NON_FALL'),
    'predicted': np.where(preds == 1, 'FALL', 'NON_FALL'),
    'fall_probability': probs.round(4),
    'correct': (preds == true),
})
pred_table


---

**Next step for SPARK proper:** once Action #23 (self-collected dataset protocol) is confirmed with the HOD, this same Stage 2 feature-engineering + Stage 4 RF/XGB pipeline can be re-run on the self-collected KEC dataset for direct comparison against SisFall-pretrained results — the `Dense(32, ReLU) -> Dense(2, Softmax)` CNN head in `train_cnn.py` remains the system's actual on-device classifier; this notebook's RF/XGB models are an assignment deliverable and an explainability/sanity-check baseline, not a replacement for it.
